# RAG System: End-to-End Demonstration

This notebook provides a complete demonstration of the Document Question Answering system built with LlamaIndex. It covers:

1. **Document Loading & Preprocessing**
2. **Index Building** with different chunking strategies
3. **Retrieval** and context selection
4. **Generation** with different LLM backends
5. **Evaluation** and result visualization

## 1. Setup and Imports

In [1]:
import sys
import os

# Add project root to path
sys.path.insert(0, os.path.abspath('..'))

import logging
from pathlib import Path

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

print("Imports successful!")

Imports successful!


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
%cd /content/drive/MyDrive/6493/project

/content/drive/MyDrive/6493/project


In [4]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -r requirements.txt
!pip install "numpy<2.0" "transformers<5.0"

Looking in indexes: https://download.pytorch.org/whl/cu121


## 2. Load Documents

In [5]:
from src.loaders import DocumentLoader

# Initialize loader
loader = DocumentLoader('data')

# Load all documents
result = loader.load_all_documents()

print(f"Loaded {len(result.documents)} documents")
print(f"Failed files: {len(result.failed_files)}")
print(f"\nAvailable documents:")
for doc in result.documents:
    print(f"  - {doc.metadata.get('file_name', 'unknown')}: {len(doc.text)} chars")

Loaded 6 documents
Failed files: 0

Available documents:
  - sample_doc1.txt: 2170 chars
  - sample_doc6.txt: 3087 chars
  - sample_doc2.txt: 2552 chars
  - sample_doc4.txt: 3314 chars
  - sample_doc3.txt: 2714 chars
  - sample_doc5.txt: 3359 chars


## 3. Build Index

In [6]:
from src.indexing import ChunkingConfig, IndexBuilder

# Choose configuration
config_path = './configs/chunking_256.yaml'

# Load configuration
config = ChunkingConfig(config_path)

print(f"Configuration:")
print(f"  Chunk size: {config.chunk_size}")
print(f"  Overlap: {config.overlap_ratio}")
print(f"  Strategy: {config.strategy}")
print(f"  Embedding: {config.embedding_model}")

Configuration:
  Chunk size: 256
  Overlap: 0.1
  Strategy: sentence
  Embedding: sentence-transformers/all-MiniLM-L6-v2


In [ ]:
# Build index
builder = IndexBuilder(config, run_id='demo_run_mistral7b')
index = builder.build_index(result.documents)

print(f"\nIndex built successfully!")
print(f"Output directory: {builder.output_dir}")

Generating embeddings:   0%|          | 0/28 [00:00<?, ?it/s]


Index built successfully!
Output directory: results/runs/demo_run_mistral7b/index


## 4. Initialize RAG Pipeline

In [ ]:
from src.rag_pipeline import RAGPipeline, RAGConfig

# Create configuration
rag_config = RAGConfig(
    llm_backend='mistral7b',  # Use mock for demo; change to 'flant5' or 'mistral7b'
    top_k=3,
    max_new_tokens=256,
    temperature=0.1
)

# Create pipeline
pipeline = RAGPipeline(index, rag_config)

print("RAG Pipeline initialized!")

RAG Pipeline initialized!


"Mock" refers to a simulated/for-testing LLM backend. The reason for choosing "mock" is to simulate responses and facilitate rapid testing.

Why use mock?

Quick verification: Without loading the real model, the test can be completed within a few seconds.

Development and debugging: Verify whether the RAG process (retrieval → generation) is functioning properly.

CI/CD: Automated testing can be conducted without the need for GPUs or large models.

## 5. Run Single Query Demo

In [ ]:
# Run a sample query
query = "What is retrieval-augmented generation (RAG)?"

print(f"Query: {query}")
print("-" * 60)

# Execute query
response = pipeline.query(query)

# Display results
print(f"\n=== ANSWER ===")
print(response.answer)

print(f"\n=== LATENCY ===")
print(f"Retrieval: {response.retrieval_latency:.3f}s")
print(f"Generation: {response.generation_latency:.3f}s")
print(f"Total: {response.total_latency:.3f}s")

Query: What is retrieval-augmented generation (RAG)?
------------------------------------------------------------

=== ANSWER ===
Retrieval-augmented generation (RAG) is a paradigm shift in how large language models (LLMs) process and generate information. It combines the power of retrieval systems with generative models to address limitations in traditional LLMs, such as their training data having a fixed cutoff date and not containing domain-specific or up-to-date information [C1]. The RAG architecture consists of three main components: a retrieval component, a generation component, and an integration component [C1]. The retrieval component searches a knowledge base to find relevant documents or text chunks using dense embeddings from transformer models to capture semantic similarity [C1]. The generation component combines the retrieved context with the original query to form a prompt, and the LLM generates a response conditioned on this retrieved information, ensuring factual accura

The time it takes to load the model for the first run into memory is very long.

The second run will then show the actual reasoning time (stored in memory)

## 6. Display Retrieved Chunks

In [ ]:
print("=== RETRIEVED CHUNKS ===")
print(f"Retrieved {len(response.retrieved_chunks)} chunks:\n")

for i, chunk in enumerate(response.retrieved_chunks, 1):
    print(f"--- Chunk {i} ---")
    print(f"ID: {chunk['chunk_id']}")
    print(f"Source: {chunk['source_file']}")
    print(f"Relevance Score: {chunk['score']:.4f}")
    print(f"Text Preview: {chunk['text_preview'][:150]}...")
    print()

=== RETRIEVED CHUNKS ===
Retrieved 3 chunks:

--- Chunk 1 ---
ID: sample_doc1_chunk_0
Source: sample_doc1.txt
Relevance Score: 0.5685
Text Preview: # Sample Document 1: Introduction to Retrieval-Augmented Generation (RAG) ## Overview Retrieval-Augmented Generation (RAG) represents a paradigm shift...

--- Chunk 2 ---
ID: sample_doc4_chunk_0
Source: sample_doc4.txt
Relevance Score: 0.4667
Text Preview: # Sample Document 4: Evaluation Metrics for RAG Systems ## Introduction Evaluating RAG systems requires measuring both the retrieval component and the...

--- Chunk 3 ---
ID: sample_doc3_chunk_3
Source: sample_doc3.txt
Relevance Score: 0.3639
Text Preview: **Memory**: Dense indices can be larger than sparse indices 4. **Hot Tokens**: Some tokens may be underrepresented in training ## Hybrid Retrieval Man...



## 7. Batch Query Evaluation

In [ ]:
from src.eval_protocol import EvaluationProtocol

# Load evaluation queries
protocol = EvaluationProtocol()
queries = protocol.load_queries('./data/queries.jsonl')

print(f"Loaded {len(queries)} evaluation queries")

# Process a few queries
sample_queries = queries[:5]
results = []

for q in sample_queries:
    print(f"\nProcessing: {q.question[:50]}...")
    response = pipeline.query(q.question)
    results.append({
        'query_id': q.id,
        'query': q.question,
        'answer': response.answer,
        'total_latency': response.total_latency,
        'num_chunks': len(response.retrieved_chunks)
    })

print(f"\nProcessed {len(results)} queries")

Loaded 20 evaluation queries

Processing: What is the main difference between BM25 and dense...

Processing: How does chunk size affect the quality of retrieve...

Processing: What are the advantages of using sentence-transfor...

Processing: Explain the role of overlap in text chunking and w...

Processing: How does Mistral-7B-Instruct compare to T5-base in...

Processed 5 queries


## 8. Results Summary

In [ ]:
import pandas as pd

# Create results DataFrame
df = pd.DataFrame(results)
df['answer_length'] = df['answer'].apply(len)

print("=== RESULTS SUMMARY ===")
print(df[['query_id', 'answer_length', 'total_latency', 'num_chunks']].to_string(index=False))

print(f"\nAverage latency: {df['total_latency'].mean():.3f}s")
print(f"Average answer length: {df['answer_length'].mean():.0f} chars")

=== RESULTS SUMMARY ===
query_id  answer_length  total_latency  num_chunks
    q001            887      70.794187           3
    q002            572      45.031979           3
    q003            823      69.264563           3
    q004            710      46.459855           3
    q005            767      55.251781           3

Average latency: 57.360s
Average answer length: 752 chars


## 9. Visualization  ***（Simulation，Non-real data ）***

If you want to use a specific LLM model, such as Mistral7b, you cannot use the following visualization code. Instead, you need to enter the following command in the command line:



```
┌─────────────────────────────────────────────────────────────┐
│  Step 1: Run the assessment (generate the initial results)                 │
│  python src/run_eval.py --llm flant5 --chunk 256            │
│  python src/run_eval.py --llm flant5 --chunk 512            │
│  python src/run_eval.py --llm mistral7b --chunk 256           │
│  python src/run_eval.py --llm mistral7b --chunk 512         │
│                 ↓                                                          │
│  Generate 4 CSV files (excluding the scores)The results are saved in "results/runs/<timestamp>/"
│ (And they need to be manually renamed later)              
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│  Step 2: Manual Scoring                                            │
│  Add "relevance_score" and "task_completion" columns to 4 CSV files    │
│  save as *_scored.csv 文件                                    │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│  Step 3: Generate visualization                                          │
│  python src/generate_visualizations.py                       │
│              ↓                                                          │
│  Generate comparison charts of real data (such as relevance_comparison.png, etc.)          │
└─────────────────────────────────────────────────────────────┘
```



**Create index**

In [ ]:
!python src/run_build_index.py --config configs/chunking_256.yaml --output-dir results/runs/run_256/index
!python src/run_build_index.py --config configs/chunking_512.yaml --output-dir results/runs/run_512/index

**Run the evaluation**

/content/drive/MyDrive/6493/project/results/runs/"timestamps" rename as eval_flant5_256

In [9]:
!python src/run_eval.py --llm flant5 --chunk 256

2026-04-12 13:27:17,890 - INFO - NumExpr defaulting to 2 threads.
2026-04-12 13:27:23.443470: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776000443.463786   16094 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776000443.470537   16094 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776000443.487751   16094 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776000443.487776   16094 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:177600044

/content/drive/MyDrive/6493/project/results/runs/"timestamps" rename as eval_flant5_512

In [8]:
!python src/run_eval.py --llm flant5 --chunk 512

2026-04-12 13:25:24,342 - INFO - NumExpr defaulting to 2 threads.
2026-04-12 13:25:29.778161: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776000329.800032   15598 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776000329.807370   15598 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776000329.826227   15598 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776000329.826253   15598 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:177600032

/content/drive/MyDrive/6493/project/results/runs/"timestamps" rename as eval_mistral7b_256

In [10]:
!python src/run_eval.py --llm mistral7b --chunk 256

2026-04-12 13:29:23,099 - INFO - NumExpr defaulting to 2 threads.
2026-04-12 13:29:29.136348: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776000569.160567   16626 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776000569.173228   16626 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776000569.194636   16626 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776000569.194660   16626 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:177600056

/content/drive/MyDrive/6493/project/results/runs/"timestamps" rename as eval_mistral7b_512

In [11]:
!python src/run_eval.py --llm mistral7b --chunk 512

2026-04-12 13:57:39,080 - INFO - NumExpr defaulting to 2 threads.
2026-04-12 13:57:47.335524: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776002267.356880   23883 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776002267.364020   23883 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776002267.381782   23883 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776002267.381808   23883 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:177600226

In [12]:
!python src/generate_visualizations.py

加载数据...
  [Flan-T5/256] 20 条记录
  [Flan-T5/512] 20 条记录
  [Mistral-7B/256] 20 条记录
  [Mistral-7B/512] 20 条记录

生成可视化图表...
/content/drive/MyDrive/6493/project/src/generate_visualizations.py:72: MatplotlibDeprecationWarning: The 'labels' parameter of boxplot() has been renamed 'tick_labels' since Matplotlib 3.9; support for the old name will be dropped in 3.11.
  bp = axes[1].boxplot(relevance_data, labels=list(configs.keys()), patch_artist=True,
  [完成] relevance_comparison.png
  [完成] completion_comparison.png
  [完成] latency_comparison.png
  [完成] response_length_comparison.png
  [完成] radar_comparison.png
  [完成] relevance_heatmap.png

生成汇总统计表...
  [完成] comparison_summary.csv
  [完成] analysis_report.txt

COMPARISON SUMMARY
 Configuration  Avg Relevance (1-5)  Relevance Std  Min Relevance  Max Relevance Task Completion Rate  Avg Words  Avg Retrieval Latency (ms)  Avg Generation Latency (s)  Avg Total Latency (s)
   Flan-T5/256                  2.1           0.72            1.0            4.0    

---

## Conclusion

This notebook demonstrated:

- Document loading and preprocessing
- Index building with configurable chunking strategies
- Retrieval-augmented generation pipeline
- Evaluation and result analysis

To run full experiments, use the CLI scripts:

```bash
# Build indices
python src/run_build_index.py --config configs/chunking_256.yaml
python src/run_build_index.py --config configs/chunking_512.yaml

# Run evaluations
python src/run_eval.py --llm flant5 --chunk 256
python src/run_eval.py --llm flant5 --chunk 512
python src/run_eval.py --llm mistral7b --chunk 256
python src/run_eval.py --llm mistral7b --chunk 512
```